# EasyOCR

In [ ]:
import easyocr
import cv2
import math
from ultralytics import YOLO 
import csv
import time
import os

# Definiciones fuera del bucle
model = YOLO("./runs/detect/train/weights/best.pt")
classes = {0:"licenseplate", 1:"car"}
reader = easyocr.Reader(['en'], gpu=True) 
capture_video = cv2.VideoCapture("video5.mp4")

# Variables para estadísticas de tiempo
tiempos_inferencia_yolo = []
tiempos_inferencia_easyocr = []

# Configuración del video de salida
frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)

# Crear VideoWriter para guardar el video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_video = cv2.VideoWriter('output_video_easyocr.mp4', fourcc, fps, (frame_width, frame_height))

# Configuración del CSV
csv_filename = "deteccion_de_matricula_easyocr.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion", "identificador_tracking", 
    "x1", "y1", "x2", "y2", "matrícula_detectada", 
    "x1_matrícula", "y1_matrícula", "x2_matrícula", "y2_matrícula", "texto_matricula_ocr",
    "tiempo_inferencia_yolo", "tiempo_inferencia_easyocr"  # Nombres más precisos
]

# Abrir archivo CSV
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';')
    csv_writer.writerow(csv_header)
    
    frame_count = 0
    
    while(True):
        ret, frame_video = capture_video.read()
        if not ret:
            break
            
        frame_count += 1
        
        # --- MEDICIÓN TIEMPO DE INFERENCIA YOLO ---
        start_time_yolo = time.perf_counter()
        results = model.track(frame_video, persist=True,conf=0.7, tracker='bytetrack.yaml', stream=True)
        end_time_yolo = time.perf_counter()
        tiempo_inferencia_yolo = end_time_yolo - start_time_yolo
        tiempos_inferencia_yolo.append(tiempo_inferencia_yolo)
        
        for frames in results:
            boxes = frames.boxes
            frame_detections = []
            
            for box in boxes:
                # Clase
                cls = int(box.cls[0])
                
                if cls not in classes.keys():
                    continue
                
                # Coordenadas
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100

                # Tracking ID
                track_id = int(box.id[0]) if box.id is not None and box.id.numel() > 0 else -1

                # Calcular colores
                escala = int((cls / len(classes)) * 255 * 3)
                if escala >= 255 * 2:
                    R, G, B = 255, 255, escala - 255 * 2
                elif escala >= 255:
                    R, G, B = 255, escala - 255, 0
                else:
                    R, G, B = escala, 0, 0
                
                # Variables para matrícula
                plate_text = ""
                tiempo_inferencia_easyocr = 0.0
                lp_x1, lp_y1, lp_x2, lp_y2 = "", "", "", ""
                
                # --- MEDICIÓN TIEMPO DE INFERENCIA EASYOCR ---
                if classes[cls] == "licenseplate":
                    lp_x1, lp_y1, lp_x2, lp_y2 = x1, y1, x2, y2
                    license_plate_img = frame_video[y1:y2, x1:x2]
                    
                    if license_plate_img.size > 0:
                        start_time_easyocr = time.perf_counter()
                        result_ocr = reader.readtext(
                            license_plate_img, 
                            allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
                            detail=0
                        )
                        end_time_easyocr = time.perf_counter()
                        tiempo_inferencia_easyocr = end_time_easyocr - start_time_easyocr
                        tiempos_inferencia_easyocr.append(tiempo_inferencia_easyocr)
                        
                        if result_ocr:
                            plate_text = "".join(result_ocr).replace(" ", "") 
                            print(f"Matrícula Detectada: {plate_text}")
                            
                            # Mostrar el texto del OCR
                            cv2.putText(
                                frame_video, 
                                plate_text, 
                                (x1, y1 - 20),
                                cv2.FONT_HERSHEY_SIMPLEX, 
                                1, 
                                (0, 255, 0),
                                2
                            )
                
                # Dibujar bounding box
                cv2.rectangle(frame_video, (x1, y1), (x2, y2), (R, G, B), 3)
                
                # Etiqueta con clase, ID y confianza
                label = f"{classes[cls]} ID:{track_id} Conf:{confidence:.2f}"
                cv2.putText(frame_video, label, [x1, y1-10], cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                # Mostrar tiempos de inferencia en pantalla
                time_info = f"Inf YOLO: {tiempo_inferencia_yolo:.3f}s"
                if tiempo_inferencia_easyocr > 0:
                    time_info += f" | EasyOCR: {tiempo_inferencia_easyocr:.3f}s"
                
                cv2.putText(frame_video, time_info, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(frame_video, f"Frame: {frame_count}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

                # Estadísticas en tiempo real
                if tiempos_inferencia_yolo:
                    avg_yolo = sum(tiempos_inferencia_yolo) / len(tiempos_inferencia_yolo)
                    avg_easyocr = sum(tiempos_inferencia_easyocr) / len(tiempos_inferencia_easyocr) if tiempos_inferencia_easyocr else 0
                    
                    stats_text = f"Avg - YOLO: {avg_yolo:.3f}s | EasyOCR: {avg_easyocr:.3f}s"
                    cv2.putText(frame_video, stats_text, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                # Preparar datos para CSV
                row_data = [
                    frame_count,
                    classes[cls],
                    confidence,
                    track_id,
                    x1, y1, x2, y2,
                    "SÍ" if classes[cls] == "licenseplate" else "NO",
                    lp_x1, lp_y1, lp_x2, lp_y2,
                    plate_text,
                    f"{tiempo_inferencia_yolo:.6f}",        # Tiempo inferencia YOLO
                    f"{tiempo_inferencia_easyocr:.6f}"      # Tiempo inferencia EasyOCR
                ]
                frame_detections.append(row_data)
            
            # Escribir todas las detecciones del frame en CSV
            csv_writer.writerows(frame_detections)
            
            # Escribir frame en video de salida
            out_video.write(frame_video)
            
            # Mostrar video en tiempo real
            cv2.imshow('Deteccion y Tracking', frame_video)
        
        # Salir con ESC
        if cv2.waitKey(20) == 27:
            break

# Liberar recursos
capture_video.release()
out_video.release()
cv2.destroyAllWindows()


print(f"\nProcesamiento completado!")
print(f"✓ Video guardado como: output_video.mp4")
print(f"✓ CSV guardado como: {csv_filename}")

# Tesseract

In [ ]:
import pytesseract
from pytesseract import Output
import cv2
import math
from ultralytics import YOLO 
from collections import defaultdict
import numpy as np 
import csv 
import os 

# --- Configuración Inicial ---
MODELO_PATH = "./runs/detect/train/weights/best.pt"
VID_PATH = "video5.mp"

# Configuración de Tesseract (Asegúrate de que la ruta sea correcta)
tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract' 
try:
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
except Exception as e:
    print(f"Advertencia: Tesseract no se pudo configurar. El OCR no funcionará. Error: {e}")

model = YOLO(MODELO_PATH) 
classes = {0: "Matricula", 1: "Coche"}
CLASSES_TO_TRACK = list(classes.keys()) 
COLOR_TEXTO_CONTEO = (0, 255, 255) # Amarillo

capture_video = cv2.VideoCapture(VID_PATH)
if not capture_video.isOpened():
    print(f"Error: No se puede abrir el video en {VID_PATH}")
    exit()

# Variables de Tracking y Conteo
contador_clases_unicas = {nombre: 0 for nombre in classes.values()}
tracker_ids_contados = set()
track_history = defaultdict(lambda: []) 
frame_count = 0 

# Obtener propiedades del video original para el video de salida
frame_width = int(capture_video.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture_video.get(cv2.CAP_PROP_FPS)

# --- Configuración del Video de Salida ---
output_video_filename = "output_conteo.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out_video = cv2.VideoWriter(
    output_video_filename, 
    fourcc, 
    fps, 
    (frame_width, frame_height)
)
print(f"Configurado VideoWriter para guardar en '{output_video_filename}'")


# --- Configuración del CSV ---
csv_filename = "conteo_y_matriculas.csv"
csv_header = [
    "fotograma", "tipo_objeto", "confianza_deteccion", "identificador_tracking", 
    "x1", "y1", "x2", "y2", "conteo_acumulado_unico_clase",
    "es_matricula", "texto_matricula_ocr" # Simplificamos las coordenadas de la matrícula, usando las mismas que la BB
]

# Abrir el archivo CSV en modo escritura y escribir la cabecera
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    csv_writer = csv.writer(csvfile, delimiter=';') 
    csv_writer.writerow(csv_header)
    
    # --- Bucle Principal de Procesamiento ---
    while(True): 
        ret, img = capture_video.read()
        if not ret: 
            break
            
        frame_count += 1
        
        results = model.track(
            img, 
            persist=True, 
            classes=CLASSES_TO_TRACK, 
            tracker="bytetrack.yaml", 
            verbose=False
        )
        
        frame_detections = [] 
        
        if results and results[0].boxes.id is not None:
            boxes_data = results[0].boxes 
            
            for box in boxes_data:
                cls = int(box.cls[0])
                conf = box.conf[0].item()
                track_id = int(box.id[0].item())
                
                x1, y1, x2, y2 = [int(val) for val in box.xyxy[0].tolist()]
                
                if cls in classes.keys():
                    clase_nombre = classes[cls]
                    
                    # 1. CONTEO ÚNICO
                    if track_id not in tracker_ids_contados:
                        contador_clases_unicas[clase_nombre] += 1
                        tracker_ids_contados.add(track_id)
                    
                    
                    # 2. DEFINIR COLOR PARA BB (usando el método original)
                    escala = int((cls / len(classes)) * 255 * 3)
                    R, G, B = (255, 255, 0) if clase_nombre == "Matricula" else (0, 0, 255) # Rojo/Azul simplificado para contraste
                    color = (B, G, R) # OpenCV es BGR
                    
                    
                    # 3. OCR (Solo para matrículas)
                    plate_text = ""
                    es_matricula_csv = "NO"
                    
                    if clase_nombre == "Matricula":
                        es_matricula_csv = "SÍ"
                        # Asegurar que las coordenadas sean válidas para el recorte
                        license_plate_img = img[max(0, y1):min(frame_height, y2), max(0, x1):min(frame_width, x2)]
                        
                        if license_plate_img.size > 0:
                            ocr_config = r'--psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                            try:
                                # Pre-procesamiento para OCR
                                lp_gray = cv2.cvtColor(license_plate_img, cv2.COLOR_BGR2GRAY)
                                _, lp_thresh = cv2.threshold(lp_gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
                                
                                plate_text_raw = pytesseract.image_to_string(lp_thresh, config=ocr_config)
                                plate_text = plate_text_raw.strip().replace(" ", "").replace("\n", "").replace("\r", "")
                            except Exception:
                                plate_text = "OCR_Error_o_NoTesseract"

                        # Dibujar texto OCR
                        if plate_text and plate_text not in ["OCR_Error_o_NoTesseract"]:
                            cv2.putText(img, plate_text, (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                            
                    
                    # 4. DIBUJAR EN EL FRAME
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                    etiqueta = f"ID:{track_id} {clase_nombre} ({conf*100:.0f}%)"
                    cv2.putText(img, etiqueta, [x1, y1 - 10], cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                    
                    # 5. DIBUJAR RASTRO (Track History)
                    center_x, center_y = (x1 + x2) // 2, (y1 + y2) // 2
                    track = track_history[track_id]
                    track.append((float(center_x), float(center_y)))
                    if len(track) > 30: 
                        track.pop(0)
                    points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(img, [points], isClosed=False, color=(230, 230, 230), thickness=5)
                    
                    
                    # 6. ALMACENAR DATOS PARA CSV
                    row_data = [
                        frame_count, 
                        clase_nombre, 
                        round(conf, 4), 
                        track_id, 
                        x1, y1, x2, y2, 
                        contador_clases_unicas[clase_nombre],
                        es_matricula_csv,
                        plate_text
                    ]
                    frame_detections.append(row_data)
            
            # Escribir todas las detecciones de este fotograma al CSV
            csv_writer.writerows(frame_detections)


        # 7. VISUALIZAR CONTEO ÚNICO
        y_offset = 30
        cv2.putText(img, "--- CONTEO UNICO ACUMULADO ---", (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        y_offset += 30
        
        for nombre, cantidad in contador_clases_unicas.items():
            texto_conteo = f"Total Unicos {nombre}: {cantidad}"
            cv2.putText(img, texto_conteo, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLOR_TEXTO_CONTEO, 2, cv2.LINE_AA) 
            y_offset += 30

        # 8. Guardar Frame y Mostrar
        out_video.write(img) 
        cv2.imshow('Deteccion, Conteo y OCR', img)
    

        if cv2.waitKey(1) == 27: 
            break 
    
# --- Liberar recursos ---
capture_video.release()
out_video.release()
cv2.destroyAllWindows()

print("\n--- RESUMEN FINAL DE CONTEO DE OBJETOS ÚNICOS ---")
for nombre, cantidad in contador_clases_unicas.items():
    print(f"Total acumulado de objetos únicos ({nombre}) detectados: {cantidad}")
print(f"Procesamiento finalizado. Video de salida guardado en '{output_video_filename}'.")
print(f"Datos guardados en '{csv_filename}'.")

Configurado VideoWriter para guardar en 'output_conteo.mp4'

--- RESUMEN FINAL DE CONTEO DE OBJETOS ÚNICOS ---
Total acumulado de objetos únicos (Matricula) detectados: 1
Total acumulado de objetos únicos (Coche) detectados: 3
Procesamiento finalizado. Video de salida guardado en 'output_conteo.mp4'.
Datos guardados en 'conteo_y_matriculas.csv'.
